# Breadth-First Search and Paths


```{contents}
:local:
:depth: 2
```


```{index} Discover a Vertex Once, Then Process Its Neighbors
```



```{index} Turn Queue Events into a Traversal Loop
```

## Turn Queue Events into a Traversal Loop


A facilities graph answers which rooms can be reached and how many doorways a shortest route uses. Every edge in this lesson contributes one hop. The queue is not a list of completed rooms; it holds discovered rooms whose neighbors still need examination. This is the queue policy from Chapter 15 applied to the graph model from the preceding sections.

Use directed edges A→B, A→C, B→D and C→D, plus an isolated E. Neighbor order is written explicitly. Begin with A discovered, distance zero and no predecessor. The small program exposes the traversal core; the full graph class and validation examples remain below.


In [1]:
using System;
using System.Collections.Generic;
var next = new Dictionary<string, string[]>
{
    ["A"] = new[] { "B", "C" }, ["B"] = new[] { "D" },
    ["C"] = new[] { "D" }, ["D"] = Array.Empty<string>(), ["E"] = Array.Empty<string>()
};
var distance = new Dictionary<string, int> { ["A"] = 0 };
var previous = new Dictionary<string, string>();
var waiting = new Queue<string>();
waiting.Enqueue("A");
while (waiting.Count > 0)
{
    string current = waiting.Dequeue();
    foreach (string neighbor in next[current])
    {
        if (distance.ContainsKey(neighbor)) continue;
        distance[neighbor] = distance[current] + 1;
        previous[neighbor] = current;
        waiting.Enqueue(neighbor);
        Console.WriteLine($"Discover {neighbor}: distance {distance[neighbor]}, from {current}");
    }
}
Console.WriteLine($"D hops: {distance["D"]}; E reached: {distance.ContainsKey("E")}");


Discover B: distance 1, from A
Discover C: distance 1, from A
Discover D: distance 2, from B
D hops: 2; E reached: False


Here membership in `distance` also means discovered, so a separate visited set would duplicate that fact. All endpoints are declared in this teaching graph. Production validation of unknown labels is retained in the complete implementation.

| Statement group | Meaning in the trace | Why the order matters |
| --- | --- | --- |
| Dequeue current | Choose the oldest waiting room | Complete this layer before later layers |
| Test distance membership | Skip a previously discovered neighbor | C must not enqueue D a second time |
| Store distance and predecessor | Record the first discovery | A known room has one chosen route witness |
| Enqueue neighbor | Schedule its remaining work | Mark discovery before another parent can find it |

After A, the queue is B,C. After B, it is C,D; D already has distance two and predecessor B. When C examines D, the membership test succeeds and no new queue entry is added. A cycle back to A is harmless for the same reason: A was marked before processing began.

A predecessor map points backwards. For D, following D←B←A gives a reverse route. Reverse those labels to display A→B→D. A missing destination has no such chain; do not create a one-room route and call it reachable. The source itself is a valid one-room route with zero edges.


```{index} Guide the Route and Tie-Policy Checks
```

## Guide the Route and Tie-Policy Checks


Fill this reverse chain before writing code:

| Current room | Recorded predecessor | Reverse route so far |
| --- | --- | --- |
| D | B | D |
| B | A | D,B |
| A | None; this is the source | D,B,A |

Guided task: change A's neighbors to C,B. Predict the new predecessor of D and the hop count. Then add D→A and check how many times A and D are discovered. Finally, ask for E and specify the result without indexing a missing map entry.

```{dropdown} Check the graph reasoning
D's predecessor becomes C, giving A→C→D, but its distance stays two. The cycle cannot rediscover A because A is already in the distance map. A and D are each discovered once. E is a known but unreachable vertex and has no route from A. Tie-dependent routes can differ while their minimum hop counts agree.
```

For independent application, use the chapter's network tasks to report both distances and route witnesses. Check each route edge and its edge count, rather than trusting one printed distance. The retained complete programs below provide the graph container, boundary checks and additional practice.


```{index} Discover a Vertex Once, Then Process Its Neighbors
```

## Discover a Vertex Once, Then Process Its Neighbors



A delivery graph has edges A-B, A-C, B-D and C-D. Starting at A, how many roads are needed to reach each stop? Breadth-first search answers a minimum-hop question in an unweighted graph. It keeps a queue of discovered vertices whose outgoing neighbors still need examination. Discovery and processing are different events: a vertex can already be known while it is still waiting in the queue.

Mark a vertex discovered when it is enqueued. If you wait until dequeue, both B and C can enqueue D before D is processed. That duplicates work and makes predecessor choices harder to reason about. A visited/discovered set is therefore part of the algorithm's state, not merely an optional speed optimization.

| Processing event | Queue afterward | Newly discovered distances |
| --- | --- | --- |
| Initialize A | A | A: 0 |
| Examine A | B, C | B: 1, C: 1 |
| Examine B | C, D | D: 2 |
| Examine C | D | None; D is already known |
| Examine D | Empty | None |

The table assumes that A lists B before C. A different neighbor order can choose another equally short predecessor route, but it does not change the minimum hop distance. Tests that insist on one exact path without specifying neighbor order can reject correct results. State the tie policy separately from the distance claim.

For related explanation, see the further reading below.[^development-reading]


```{index} Recover a Path from Earlier Layers
```



```{index} Recover a Path from Earlier Layers
```

## Recover a Path from Earlier Layers



When discovering a new vertex v from u, record predecessor[v] = u and distance[v] = distance[u] + 1. Following those predecessor links backward reconstructs a route to the source. Reverse the route to display it from source to destination. The source is a one-vertex route to itself; an unreachable known vertex has no source route. Do not fabricate a destination-only path when no predecessor chain exists.

Why is the distance minimal? The queue processes vertices in nondecreasing discovered distance. All one-hop vertices are discovered from the source before processing can expose a two-hop successor, and the same reasoning repeats layer by layer. The first discovery therefore uses a shortest hop count. This reasoning relies on equal edge contribution, not on geometric closeness or vertex spelling. With arbitrary weights, fewer edges can be more expensive; Chapter 24 treats that different objective.

For adjacency lists, visiting reachable vertices and their incident/outgoing entries costs work proportional to those entries plus vertex bookkeeping. A traversal of one component does not visit all disconnected components by magic. If the task is a full undirected component report, start a new traversal from each still-undiscovered vertex, including isolates.

Guided exercise: add an isolated E and a cycle D-A to the graph. Predict which vertices are discovered from A, whether any is enqueued twice, and the result for a path to E. Then choose C before B in A's neighbor list. Explain which outputs may change and which correctness obligations remain fixed.

```{dropdown} Verification checkpoint
E is unreachable from A; its route must remain absent. The cycle does not rediscover A because A was marked at initialization. Either A-B-D or A-C-D can be a shortest route to D under the chosen neighbor order; both use two edges.
```


```{index} Graph Traversal: Reachability, Paths, and Components
```

<span id="graph-traversal-reachability-paths-and-components"></span>


A facilities team needs to know which rooms can be reached, how many unweighted links a route needs, and which rooms form disconnected groups. A representation stores connections; traversal systematically discovers what those connections imply. Start with declared vertices so isolated rooms remain visible. Every example includes the complete model. The implementation uses ordinary classes, loops, a queue, and a visited set.


```{index} Breadth-First Search and Discovery
```

## Breadth-First Search and Discovery




```{index} Breadth-First Search and Discovery
```


Breadth-first search (BFS) discovers vertices in layers. Mark a vertex when it enters the queue, so two incoming edges cannot enqueue it twice. The source has distance zero. Each newly discovered neighbor receives its predecessor distance plus one. Queue order ensures that all vertices at distance d are processed before distance d+1. These are minimum edge counts for an unweighted graph; they are not minimum travel times when weights differ.


In [1]:
using System;
using System.Collections.Generic;

SimpleGraph graph = new SimpleGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Remote" }) graph.AddVertex(room);
graph.AddEdge("Lobby", "Hall"); graph.AddEdge("Hall", "Lab");
graph.AddEdge("Hall", "Office"); graph.AddEdge("Office", "Lab");
SearchResult result = graph.Bfs("Lobby");
foreach (string room in graph.Vertices())
    Console.WriteLine($"{room}: {result.DistanceTo(room)?.ToString() ?? "unreachable"}");

public class SimpleGraph
{
    private readonly Dictionary<string, List<string>> adjacency
        = new Dictionary<string, List<string>>(StringComparer.Ordinal);
    private readonly List<string> vertexOrder = new List<string>();
    public bool IsDirected { get; }
    public SimpleGraph(bool directed = false) { IsDirected = directed; }
    public void AddVertex(string vertex)
    {
        if (string.IsNullOrWhiteSpace(vertex)) throw new ArgumentException("Vertex required.");
        if (adjacency.ContainsKey(vertex)) return;
        adjacency.Add(vertex, new List<string>());
        vertexOrder.Add(vertex);
    }
    public void AddEdge(string from, string to)
    {
        RequireVertex(from); RequireVertex(to);
        if (!adjacency[from].Contains(to)) adjacency[from].Add(to);
        if (!IsDirected && !adjacency[to].Contains(from)) adjacency[to].Add(from);
    }
    public string[] Vertices() { return vertexOrder.ToArray(); }
    public string[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }
    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new ArgumentException("Unknown vertex: " + vertex);
    }
    public SearchResult Bfs(string source)
    {
        RequireVertex(source);
        var distance = new Dictionary<string, int>(StringComparer.Ordinal);
        var previous = new Dictionary<string, string?>(StringComparer.Ordinal);
        var order = new List<string>();
        var waiting = new Queue<string>();
        distance.Add(source, 0); previous.Add(source, null); waiting.Enqueue(source);
        while (waiting.Count > 0)
        {
            string current = waiting.Dequeue();
            order.Add(current);
            foreach (string next in adjacency[current])
            {
                if (distance.ContainsKey(next)) continue;
                distance.Add(next, distance[current] + 1);
                previous.Add(next, current);
                waiting.Enqueue(next);
            }
        }
        return new SearchResult(distance, previous, order);
    }
    public string[] Dfs(string source)
    {
        RequireVertex(source);
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var order = new List<string>();
        Visit(source, seen, order);
        return order.ToArray();
    }
    private void Visit(string current, HashSet<string> seen, List<string> order)
    {
        if (!seen.Add(current)) return;
        order.Add(current);
        foreach (string next in adjacency[current]) Visit(next, seen, order);
    }
    public List<string[]> Components()
    {
        if (IsDirected) throw new InvalidOperationException("Components requires an undirected graph.");
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var groups = new List<string[]>();
        foreach (string vertex in vertexOrder)
        {
            if (seen.Contains(vertex)) continue;
            string[] group = Bfs(vertex).VisitOrder();
            foreach (string member in group) seen.Add(member);
            groups.Add(group);
        }
        return groups;
    }
}

public class SearchResult
{
    private readonly Dictionary<string, int> distance;
    private readonly Dictionary<string, string?> previous;
    private readonly List<string> order;
    internal SearchResult(Dictionary<string, int> distance,
        Dictionary<string, string?> previous, List<string> order)
    {
        this.distance = distance; this.previous = previous; this.order = order;
    }
    public int? DistanceTo(string target)
    {
        return distance.TryGetValue(target, out int value) ? value : null;
    }
    public string[] VisitOrder() { return order.ToArray(); }
    public string[] PathTo(string target)
    {
        if (!distance.ContainsKey(target)) return Array.Empty<string>();
        var reverse = new List<string>();
        string? current = target;
        while (current is not null)
        {
            reverse.Add(current);
            current = previous[current];
        }
        reverse.Reverse();
        return reverse.ToArray();
    }
}


Lobby: 0
Hall: 1
Lab: 2
Office: 2
Remote: unreachable


```{index} Recover a Path with Predecessors
```

## Recover a Path with Predecessors




```{index} Recover a Path with Predecessors
```


Store a predecessor only at first discovery. To recover a route, follow predecessors from the destination back to the source and reverse the collected list. The source route contains the source itself; an unreachable target produces an empty array. Neighbor insertion order chooses among equally short routes. Test the distance and valid edges instead of assuming an arbitrary tie order is a mathematical requirement.


In [1]:
using System;
using System.Collections.Generic;

SimpleGraph graph = new SimpleGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Remote" }) graph.AddVertex(room);
graph.AddEdge("Lobby", "Hall"); graph.AddEdge("Hall", "Lab");
graph.AddEdge("Hall", "Office"); graph.AddEdge("Office", "Lab");
SearchResult result = graph.Bfs("Lobby");
Console.WriteLine(string.Join(" -> ", result.PathTo("Lab")));
Console.WriteLine($"Remote route length: {result.PathTo("Remote").Length}");
Console.WriteLine(string.Join(" -> ", result.PathTo("Lobby")));

public class SimpleGraph
{
    private readonly Dictionary<string, List<string>> adjacency
        = new Dictionary<string, List<string>>(StringComparer.Ordinal);
    private readonly List<string> vertexOrder = new List<string>();
    public bool IsDirected { get; }
    public SimpleGraph(bool directed = false) { IsDirected = directed; }
    public void AddVertex(string vertex)
    {
        if (string.IsNullOrWhiteSpace(vertex)) throw new ArgumentException("Vertex required.");
        if (adjacency.ContainsKey(vertex)) return;
        adjacency.Add(vertex, new List<string>());
        vertexOrder.Add(vertex);
    }
    public void AddEdge(string from, string to)
    {
        RequireVertex(from); RequireVertex(to);
        if (!adjacency[from].Contains(to)) adjacency[from].Add(to);
        if (!IsDirected && !adjacency[to].Contains(from)) adjacency[to].Add(from);
    }
    public string[] Vertices() { return vertexOrder.ToArray(); }
    public string[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }
    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new ArgumentException("Unknown vertex: " + vertex);
    }
    public SearchResult Bfs(string source)
    {
        RequireVertex(source);
        var distance = new Dictionary<string, int>(StringComparer.Ordinal);
        var previous = new Dictionary<string, string?>(StringComparer.Ordinal);
        var order = new List<string>();
        var waiting = new Queue<string>();
        distance.Add(source, 0); previous.Add(source, null); waiting.Enqueue(source);
        while (waiting.Count > 0)
        {
            string current = waiting.Dequeue();
            order.Add(current);
            foreach (string next in adjacency[current])
            {
                if (distance.ContainsKey(next)) continue;
                distance.Add(next, distance[current] + 1);
                previous.Add(next, current);
                waiting.Enqueue(next);
            }
        }
        return new SearchResult(distance, previous, order);
    }
    public string[] Dfs(string source)
    {
        RequireVertex(source);
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var order = new List<string>();
        Visit(source, seen, order);
        return order.ToArray();
    }
    private void Visit(string current, HashSet<string> seen, List<string> order)
    {
        if (!seen.Add(current)) return;
        order.Add(current);
        foreach (string next in adjacency[current]) Visit(next, seen, order);
    }
    public List<string[]> Components()
    {
        if (IsDirected) throw new InvalidOperationException("Components requires an undirected graph.");
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var groups = new List<string[]>();
        foreach (string vertex in vertexOrder)
        {
            if (seen.Contains(vertex)) continue;
            string[] group = Bfs(vertex).VisitOrder();
            foreach (string member in group) seen.Add(member);
            groups.Add(group);
        }
        return groups;
    }
}

public class SearchResult
{
    private readonly Dictionary<string, int> distance;
    private readonly Dictionary<string, string?> previous;
    private readonly List<string> order;
    internal SearchResult(Dictionary<string, int> distance,
        Dictionary<string, string?> previous, List<string> order)
    {
        this.distance = distance; this.previous = previous; this.order = order;
    }
    public int? DistanceTo(string target)
    {
        return distance.TryGetValue(target, out int value) ? value : null;
    }
    public string[] VisitOrder() { return order.ToArray(); }
    public string[] PathTo(string target)
    {
        if (!distance.ContainsKey(target)) return Array.Empty<string>();
        var reverse = new List<string>();
        string? current = target;
        while (current is not null)
        {
            reverse.Add(current);
            current = previous[current];
        }
        reverse.Reverse();
        return reverse.ToArray();
    }
}


Lobby -> Hall -> Lab
Remote route length: 0
Lobby


```{rubric} Footnotes
```


[^development-reading]: Further reading: [OpenDSA, Graph Traversals](https://opendsa.org/OpenDSA/Books/Catalog/html/GraphTraversal.html). The worked business scenarios and guided reasoning here are original teaching examples.
